# 04 LLM-Inspired Intent Classification

This notebook is the fourth step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

The goal of this stage is to classify user purchase intent from customer review text and business signals.

In a real TikTok LIVE or SMB lead generation scenario, user comments, chat messages, product inquiries, and review text can provide valuable signals about user intent. For example, users may express price sensitivity, delivery concerns, product quality concerns, comparison shopping behaviour, or readiness to purchase.

Instead of relying on a paid LLM API, this notebook implements a reproducible LLM-inspired intent classification workflow using:

1. Customer review text;
2. Review score;
3. Delivery delay signal;
4. Product price band;
5. Business rules and keyword patterns.

The output will be a structured intent dataset that includes:

- sentiment;
- purchase_intent;
- intent_category;
- lead_score.

This dataset will later support lead scoring, recommendation strategy design, and A/B test simulation.

## Step 1: Import Libraries and Define Project Paths

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Set random seed for reproducibility
np.random.seed(42)

# Define project paths
BASE_DIR = Path("..")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
FINAL_DIR = BASE_DIR / "data" / "final"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"

# Create folders if they do not exist
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project paths defined successfully.")

Libraries imported and project paths defined successfully.


## Step 2: Load Cleaned Order Dataset and User Event Dataset

In [2]:
# Load cleaned order-level dataset
order_base = pd.read_csv(PROCESSED_DIR / "clean_order_base.csv")

# Load simulated user event log
fact_user_events = pd.read_csv(FINAL_DIR / "fact_user_events.csv")

# Convert datetime columns
order_base["order_time"] = pd.to_datetime(order_base["order_time"])
fact_user_events["event_time"] = pd.to_datetime(fact_user_events["event_time"])

print("Datasets loaded successfully.")
print("Order base shape:", order_base.shape)
print("User event log shape:", fact_user_events.shape)

Datasets loaded successfully.
Order base shape: (112650, 32)
User event log shape: (386012, 15)


## Step 3: Prepare Review-Level Dataset

In this step, I create a review-level analytical dataset. Each row represents an order with available review and product context.

Relevant fields include:

- user ID;
- order ID;
- product ID;
- seller ID;
- category;
- review score;
- review title;
- review message;
- price band;
- delivery delay;
- GMV.

This dataset will be used for intent classification.

In [4]:
# Select fields required for intent classification
reviews_base = order_base[[
    "order_id",
    "user_id",
    "product_id",
    "seller_id",
    "category",
    "order_time",
    "price",
    "price_band",
    "gmv",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "delivery_delay_days",
    "late_delivery_flag"
]].copy()

# Create a combined review text field
reviews_base["review_comment_title"] = reviews_base["review_comment_title"].fillna("")
reviews_base["review_comment_message"] = reviews_base["review_comment_message"].fillna("")

reviews_base["review_text"] = (
    reviews_base["review_comment_title"].astype(str) 
    + " " 
    + reviews_base["review_comment_message"].astype(str)
).str.strip()

# Replace empty review text with a placeholder
reviews_base["review_text"] = reviews_base["review_text"].replace("", "no_text_review")

reviews_base.head()

,order_id,user_id,product_id,seller_id,category,order_time,price,price_band,gmv,review_score,review_comment_title,review_comment_message,delivery_delay_days,late_delivery_flag,review_text
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,2017-09-13 08:59:02,58.90,Medium,72.19,5.0,,"Perfeito, produto entregue antes do combinado.",-9.0,0,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,2017-04-26 10:53:06,239.90,Premium,259.83,4.0,,,-3.0,0,no_text_review
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,2018-01-14 14:33:31,199.00,Premium,216.87,5.0,,Chegou antes do prazo previsto e o produto sur...,-14.0,0,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,2018-08-08 10:00:35,12.99,Low,25.78,4.0,,,-6.0,0,no_text_review
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,2017-02-04 13:57:51,199.90,Premium,218.04,5.0,,Gostei pois veio no prazo determinado .,-16.0,0,Gostei pois veio no prazo determinado .


## Step 4: Add User Behaviour Features

Intent classification should not rely only on text. In a product analytics context, user behaviour also provides important signals.

From the simulated user event log, we create user-order-level behaviour features:

- viewed flag;
- clicked flag;
- add-to-cart flag;
- inquiry flag;
- purchased flag;
- number of events.

These features help make lead scoring more connected to user interaction behaviour.

In [5]:
# Create user-order level behaviour features from the event log
event_features = (
    fact_user_events
    .groupby(["user_id", "order_id"])
    .agg(
        total_events=("event_type", "count"),
        viewed=("event_type", lambda x: int("view" in set(x))),
        clicked=("event_type", lambda x: int("click" in set(x))),
        added_to_cart=("event_type", lambda x: int("add_to_cart" in set(x))),
        inquired=("event_type", lambda x: int("inquiry" in set(x))),
        purchased=("event_type", lambda x: int("purchase" in set(x))),
        traffic_source=("traffic_source", "first"),
        device_type=("device_type", "first"),
        user_value_segment=("user_value_segment", "first")
    )
    .reset_index()
)

event_features.head()

,user_id,order_id,total_events,viewed,clicked,added_to_cart,inquired,purchased,traffic_source,device_type,user_value_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,4,1,1,0,1,1,Shop Tab,Android,Medium Value
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,3,1,0,0,1,1,Live Stream,iOS,Low Value
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2,1,0,0,0,1,Live Stream,Android,Medium Value
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,3,1,0,0,1,1,For You Feed,iOS,Low Value
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,3,1,1,0,0,1,Search,iOS,High Value


## Step 5: Merge Review Data with Behaviour Features

I merge review-level data with user interaction features. This creates an enriched dataset that combines:

1. Text signals;
2. Review score signals;
3. Product and seller context;
4. Delivery experience;
5. User behaviour signals.

In [6]:
# Merge review dataset with behaviour features
intent_base = reviews_base.merge(
    event_features,
    on=["user_id", "order_id"],
    how="left"
)

# Fill missing behaviour fields
behaviour_cols = ["total_events", "viewed", "clicked", "added_to_cart", "inquired", "purchased"]

for col in behaviour_cols:
    intent_base[col] = intent_base[col].fillna(0)

intent_base["traffic_source"] = intent_base["traffic_source"].fillna("Unknown")
intent_base["device_type"] = intent_base["device_type"].fillna("Unknown")
intent_base["user_value_segment"] = intent_base["user_value_segment"].fillna("Unknown")

intent_base.head()

,order_id,user_id,product_id,seller_id,category,order_time,price,price_band,gmv,review_score,...,review_text,total_events,viewed,clicked,added_to_cart,inquired,purchased,traffic_source,device_type,user_value_segment
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,cool_stuff,2017-09-13 08:59:02,58.90,Medium,72.19,5.0,...,"Perfeito, produto entregue antes do combinado.",5.0,1.0,1.0,1.0,1.0,1.0,Creator Profile,iOS,Low Value
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop,2017-04-26 10:53:06,239.90,Premium,259.83,4.0,...,no_text_review,3.0,1.0,0.0,0.0,1.0,1.0,Search,Web,High Value
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,furniture_decor,2018-01-14 14:33:31,199.00,Premium,216.87,5.0,...,Chegou antes do prazo previsto e o produto sur...,5.0,1.0,1.0,1.0,1.0,1.0,Search,Web,High Value
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,perfumery,2018-08-08 10:00:35,12.99,Low,25.78,4.0,...,no_text_review,2.0,1.0,0.0,0.0,0.0,1.0,Live Stream,iOS,Low Value
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,garden_tools,2017-02-04 13:57:51,199.90,Premium,218.04,5.0,...,Gostei pois veio no prazo determinado .,5.0,1.0,1.0,1.0,1.0,1.0,Creator Profile,iOS,High Value


## Step 6: Define Keyword Dictionaries

This step defines keyword dictionaries for intent classification. Because the Olist review text is mostly in Portuguese, the keyword list includes common Portuguese and English words related to:

- delivery concern;
- product quality concern;
- price sensitivity;
- positive sentiment;
- negative sentiment;
- after-sales issues.

This is a simplified but reproducible substitute for LLM-based text classification.

In [7]:
# Keyword dictionaries for rule-based intent classification

positive_keywords = [
    "bom", "boa", "ótimo", "otimo", "excelente", "perfeito", "perfeita",
    "gostei", "recomendo", "satisfeito", "satisfeita", "adorei",
    "good", "great", "excellent", "perfect", "recommend", "satisfied", "love"
]

negative_keywords = [
    "ruim", "péssimo", "pessimo", "horrível", "horrivel", "defeito",
    "problema", "quebrado", "danificado", "não gostei", "nao gostei",
    "bad", "poor", "terrible", "broken", "damaged", "problem", "defective"
]

delivery_keywords = [
    "atraso", "atrasado", "demora", "demorou", "entrega", "frete",
    "prazo", "correios", "delivery", "late", "delayed", "shipping", "slow"
]

quality_keywords = [
    "qualidade", "material", "defeito", "quebrado", "danificado",
    "falso", "tamanho", "cor", "produto errado",
    "quality", "material", "broken", "damaged", "wrong product", "size", "color"
]

price_keywords = [
    "preço", "preco", "caro", "barato", "valor", "desconto",
    "price", "expensive", "cheap", "discount", "value"
]

after_sales_keywords = [
    "troca", "devolução", "devolucao", "reembolso", "garantia",
    "atendimento", "suporte",
    "return", "refund", "exchange", "warranty", "support", "service"
]

comparison_keywords = [
    "comparar", "comparação", "comparacao", "melhor que", "pior que",
    "compare", "comparison", "better than", "worse than", "alternative"
]

## Step 7: Create Text Matching Helper Function

This helper function checks whether a review text contains any keywords from a given keyword list.

The function is case-insensitive and handles missing text safely.

In [8]:
def contains_keyword(text, keywords):
    """
    Check whether the input text contains any keyword from the keyword list.
    
    Parameters:
    text: str
        Review text.
    keywords: list
        List of keywords to search for.
        
    Returns:
    bool
        True if any keyword is found, otherwise False.
    """
    if pd.isna(text):
        return False
    
    text = str(text).lower()
    
    for keyword in keywords:
        if keyword.lower() in text:
            return True
    
    return False

## Step 8: Classify Sentiment

Sentiment is classified using both review score and text keywords.

Rules:

- Positive: high review score or positive keywords;
- Negative: low review score, negative keywords, or late delivery with low review score;
- Neutral: otherwise.

This creates a simple structured sentiment field for downstream analysis.

In [9]:
def classify_sentiment(row):
    """
    Classify sentiment using review score and review text.
    """
    text = row["review_text"]
    score = row["review_score"]
    
    has_positive = contains_keyword(text, positive_keywords)
    has_negative = contains_keyword(text, negative_keywords)
    
    if score >= 4 or has_positive:
        return "positive"
    elif score <= 2 or has_negative:
        return "negative"
    else:
        return "neutral"

intent_base["sentiment"] = intent_base.apply(classify_sentiment, axis=1)

intent_base["sentiment"].value_counts()

sentiment
positive    88201
negative    16510
neutral      7939
Name: count, dtype: int64

## Step 9: Classify Intent Category
Intent category describes the main reason behind the user's feedback or behaviour.

Categories:

- `ready_to_purchase`
- `delivery_concern`
- `product_quality_concern`
- `price_sensitive`
- `after_sales_issue`
- `comparison_shopping`
- `general_positive`
- `general_negative`
- `neutral_or_unclear`

This field helps transform unstructured review text into product strategy signals.

In [10]:
def classify_intent_category(row):
    """
    Classify the user's main intent category using text, score, price, and delivery signals.
    """
    text = row["review_text"]
    score = row["review_score"]
    price_band = row["price_band"]
    late_delivery = row["late_delivery_flag"]
    
    # Delivery issue has high priority if delivery was late or related keywords appear
    if contains_keyword(text, delivery_keywords) or late_delivery == 1:
        return "delivery_concern"
    
    # Product quality concern
    if contains_keyword(text, quality_keywords):
        return "product_quality_concern"
    
    # After-sales issue
    if contains_keyword(text, after_sales_keywords):
        return "after_sales_issue"
    
    # Price sensitivity
    if contains_keyword(text, price_keywords) or price_band == "Premium":
        return "price_sensitive"
    
    # Comparison shopping
    if contains_keyword(text, comparison_keywords):
        return "comparison_shopping"
    
    # Ready to purchase / satisfied user signal
    if score >= 4 and row["sentiment"] == "positive":
        return "ready_to_purchase"
    
    # General negative experience
    if score <= 2 and row["sentiment"] == "negative":
        return "general_negative"
    
    # General positive experience
    if score == 3:
        return "neutral_or_unclear"
    
    return "neutral_or_unclear"

intent_base["intent_category"] = intent_base.apply(classify_intent_category, axis=1)

intent_base["intent_category"].value_counts()

intent_category
ready_to_purchase          51869
delivery_concern           21995
price_sensitive            20822
general_negative            6117
neutral_or_unclear          5883
product_quality_concern     4883
after_sales_issue           1057
comparison_shopping           24
Name: count, dtype: int64

## Step 10: Classify Purchase Intent

Purchase intent indicates how likely the user is to become a high-quality lead.

Rules:

- High intent:
  - positive sentiment;
  - high review score;
  - inquiry or add-to-cart behaviour;
  - ready-to-purchase category.

- Medium intent:
  - neutral sentiment;
  - some engagement behaviour;
  - price sensitivity or comparison shopping.

- Low intent:
  - negative sentiment;
  - low review score;
  - delivery, product quality, or after-sales concerns.

This field will be used in the lead scoring and recommendation strategy stages.

In [12]:
def classify_purchase_intent(row):
    """
    Classify purchase intent level using sentiment, review score, intent category, and behaviour signals.
    """
    score = row["review_score"]
    sentiment = row["sentiment"]
    intent_category = row["intent_category"]
    inquired = row["inquired"]
    added_to_cart = row["added_to_cart"]
    clicked = row["clicked"]
    
    if (
        sentiment == "positive"
        and score >= 4
        and intent_category in ["ready_to_purchase", "general_positive"]
    ):
        return "high"
    
    if (
        inquired == 1
        or added_to_cart == 1
        or (clicked == 1 and sentiment == "positive")
    ):
        return "high"
    
    if intent_category in ["price_sensitive", "comparison_shopping"]:
        return "medium"
    
    if sentiment == "neutral" or score == 3:
        return "medium"
    
    if intent_category in [
        "delivery_concern",
        "product_quality_concern",
        "after_sales_issue",
        "general_negative"
    ]:
        return "low"
    
    if sentiment == "negative" or score <= 2:
        return "low"
    
    return "medium"

intent_base["purchase_intent"] = intent_base.apply(classify_purchase_intent, axis=1)

intent_base["purchase_intent"].value_counts()

purchase_intent
high      99674
low        7591
medium     5385
Name: count, dtype: int64

## Step 11: Create Lead Score

Lead score is a numeric indicator from 0 to 100.

It combines:

1. Sentiment;
2. Purchase intent;
3. Behaviour engagement;
4. Review score;
5. Delivery experience;
6. Price sensitivity.

The score is not meant to be a perfect prediction model at this stage. Instead, it provides a transparent and interpretable scoring logic for identifying high-intent leads.

In [13]:
def calculate_lead_score(row):
    """
    Calculate an interpretable lead score between 0 and 100.
    """
    score = 50
    
    # Sentiment contribution
    if row["sentiment"] == "positive":
        score += 15
    elif row["sentiment"] == "negative":
        score -= 15
    
    # Purchase intent contribution
    if row["purchase_intent"] == "high":
        score += 20
    elif row["purchase_intent"] == "medium":
        score += 5
    elif row["purchase_intent"] == "low":
        score -= 15
    
    # Review score contribution
    score += (row["review_score"] - 3) * 5
    
    # Behaviour contribution
    if row["clicked"] == 1:
        score += 5
    if row["added_to_cart"] == 1:
        score += 10
    if row["inquired"] == 1:
        score += 12
    if row["purchased"] == 1:
        score += 8
    
    # Delivery issue penalty
    if row["late_delivery_flag"] == 1:
        score -= 8
    
    # Price sensitivity penalty
    if row["intent_category"] == "price_sensitive":
        score -= 5
    
    # Product quality or after-sales issue penalty
    if row["intent_category"] in ["product_quality_concern", "after_sales_issue"]:
        score -= 10
    
    # Keep score between 0 and 100
    score = max(0, min(100, score))
    
    return round(score, 2)

intent_base["lead_score"] = intent_base.apply(calculate_lead_score, axis=1)

intent_base[[
    "user_id",
    "order_id",
    "review_score",
    "sentiment",
    "intent_category",
    "purchase_intent",
    "lead_score"
]].head(10)

,user_id,order_id,review_score,sentiment,intent_category,purchase_intent,lead_score
0,871766c5855e863f6eccc05f988b23cb,00010242fe8c5a6d1ba2dd792cb16214,5.0,positive,ready_to_purchase,high,100.0
1,eb28e67c4c0b83846050ddfb8a35d051,00018f77f2f0320c557190d7a144bdd3,4.0,positive,price_sensitive,high,100.0
2,3818d81c6709e39d06b2738a8d3a2474,000229ec398224ef6ca0657da4fc703e,5.0,positive,delivery_concern,high,100.0
3,af861d436cfc08b2c2ddefd0ba074622,00024acbcdf0a6daa1e931b038114c75,4.0,positive,ready_to_purchase,high,98.0
4,64b576fb70d441e8f1b2d7d446e483c5,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,positive,delivery_concern,high,100.0
5,85c835d128beae5b4ce8602c491bf385,00048cc3ae777c65dbb7d2a0634bc1ea,4.0,positive,ready_to_purchase,high,100.0
6,635d9ac1680f03288e72ada3a1035803,00054e8431b9d7675808bcb819fb4a32,4.0,positive,ready_to_purchase,high,100.0
7,fda4476abb6307ab3c415b7e6d026526,000576fe39319847cbb9d288c5617fa6,5.0,positive,price_sensitive,high,100.0
8,639d23421f5517f69d0c3d6e6564cf0e,0005a1a1728c9d785b8e2b08b904576c,1.0,negative,price_sensitive,medium,38.0
9,0782c41380992a5a533489063df0eef6,0005f50442cb953dcd1d21e1fb923495,4.0,positive,ready_to_purchase,high,100.0


## Step 12: Create High-Intent Lead Flag

A high-intent lead is defined as a user-order record with:

- lead score greater than or equal to 75; or
- purchase intent classified as high.

This binary flag will be useful for lead scoring, recommendation strategy, and A/B test simulation.

In [15]:
# Create high-intent lead flag
intent_base["high_intent_flag"] = np.where(
    (intent_base["lead_score"] >= 75) | (intent_base["purchase_intent"] == "high"),
    1,
    0
)

intent_base["high_intent_flag"].value_counts()

high_intent_flag
1    101106
0     11544
Name: count, dtype: int64

In [16]:
print("Intent dataset shape:", intent_base.shape)

print("\nSentiment distribution:")
print(intent_base["sentiment"].value_counts(normalize=True).round(4))

print("\nPurchase intent distribution:")
print(intent_base["purchase_intent"].value_counts(normalize=True).round(4))

print("\nIntent category distribution:")
print(intent_base["intent_category"].value_counts(normalize=True).round(4))

print("\nLead score summary:")
print(intent_base["lead_score"].describe())

Intent dataset shape: (112650, 29)

Sentiment distribution:
sentiment
positive    0.7830
negative    0.1466
neutral     0.0705
Name: proportion, dtype: float64

Purchase intent distribution:
purchase_intent
high      0.8848
low       0.0674
medium    0.0478
Name: proportion, dtype: float64

Intent category distribution:
intent_category
ready_to_purchase          0.4604
delivery_concern           0.1953
price_sensitive            0.1848
general_negative           0.0543
neutral_or_unclear         0.0522
product_quality_concern    0.0433
after_sales_issue          0.0094
comparison_shopping        0.0002
Name: proportion, dtype: float64

Lead score summary:
count    112650.000000
mean         89.973569
std          21.910071
min           0.000000
25%          98.000000
50%         100.000000
75%         100.000000
max         100.000000
Name: lead_score, dtype: float64


## Step 14: Create Intent Distribution Summary

This output summarises how many records fall into each sentiment and purchase intent group.

It helps describe the overall lead quality profile of the marketplace.

In [17]:
intent_distribution = (
    intent_base
    .groupby(["sentiment", "purchase_intent"])
    .agg(
        records=("order_id", "count"),
        unique_users=("user_id", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        high_intent_leads=("high_intent_flag", "sum")
    )
    .reset_index()
)

intent_distribution["avg_lead_score"] = intent_distribution["avg_lead_score"].round(2)

intent_distribution

,sentiment,purchase_intent,records,unique_users,avg_lead_score,high_intent_leads
0,negative,high,9561,6413,68.22,9561
1,negative,low,5926,5345,15.46,0
2,negative,medium,1023,932,33.52,0
3,neutral,high,5184,4144,92.87,5184
4,neutral,medium,2755,2612,63.38,0
5,positive,high,84929,73682,99.78,84929
6,positive,low,1665,1639,59.62,0
7,positive,medium,1607,1596,79.58,1432


In [18]:
intent_distribution.to_csv(TABLE_OUTPUT_DIR / "intent_distribution.csv", index=False)

print("intent_distribution.csv exported successfully.")

intent_distribution.csv exported successfully.


## Step 15: Create Intent Category Summary

This table summarises lead quality by intent category.
Metrics include:

- Number of records;
- Unique users;
- Average lead score;
- High-intent lead rate;
- Average GMV.

This helps identify which intent categories are more commercially valuable.

In [19]:
intent_category_summary = (
    intent_base
    .groupby("intent_category")
    .agg(
        records=("order_id", "count"),
        unique_users=("user_id", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        high_intent_leads=("high_intent_flag", "sum"),
        avg_gmv=("gmv", "mean"),
        total_gmv=("gmv", "sum")
    )
    .reset_index()
)

intent_category_summary["avg_lead_score"] = intent_category_summary["avg_lead_score"].round(2)
intent_category_summary["avg_gmv"] = intent_category_summary["avg_gmv"].round(2)
intent_category_summary["total_gmv"] = intent_category_summary["total_gmv"].round(2)

intent_category_summary["high_intent_rate"] = (
    intent_category_summary["high_intent_leads"] / intent_category_summary["records"]
).round(4)

intent_category_summary = intent_category_summary.sort_values(
    by="avg_lead_score",
    ascending=False
)

intent_category_summary

,intent_category,records,unique_users,avg_lead_score,high_intent_leads,avg_gmv,total_gmv,high_intent_rate
7,ready_to_purchase,51869,44842,99.93,51869,77.49,4019070.26,1.0000
5,price_sensitive,20822,19145,92.48,19280,321.24,6688908.40,0.9259
1,comparison_shopping,24,20,91.67,21,85.86,2060.70,0.8750
4,neutral_or_unclear,5883,4860,85.69,4117,76.82,451913.97,0.6998
2,delivery_concern,21995,19155,80.48,17573,152.97,3364614.76,0.7990
6,product_quality_concern,4883,4082,74.79,3812,136.65,667262.99,0.7807
0,after_sales_issue,1057,886,70.49,796,171.78,181570.78,0.7531
3,general_negative,6117,4212,50.81,3638,76.53,468151.38,0.5947


In [20]:
intent_category_summary.to_csv(TABLE_OUTPUT_DIR / "intent_category_summary.csv", index=False)

print("intent_category_summary.csv exported successfully.")

intent_category_summary.csv exported successfully.


## Step 16: Lead Score by Product Category

This analysis evaluates lead quality across product categories.

It helps identify which categories contain more high-intent leads and may be more suitable for recommendation exposure.

In [21]:
lead_score_by_category = (
    intent_base
    .groupby("category")
    .agg(
        records=("order_id", "count"),
        unique_users=("user_id", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        high_intent_leads=("high_intent_flag", "sum"),
        avg_gmv=("gmv", "mean"),
        total_gmv=("gmv", "sum")
    )
    .reset_index()
)

lead_score_by_category["avg_lead_score"] = lead_score_by_category["avg_lead_score"].round(2)
lead_score_by_category["avg_gmv"] = lead_score_by_category["avg_gmv"].round(2)
lead_score_by_category["total_gmv"] = lead_score_by_category["total_gmv"].round(2)

lead_score_by_category["high_intent_rate"] = (
    lead_score_by_category["high_intent_leads"] / lead_score_by_category["records"]
).round(4)

# Keep categories with enough records
lead_score_by_category = lead_score_by_category[
    lead_score_by_category["records"] >= 50
].sort_values(
    by=["high_intent_rate", "total_gmv"],
    ascending=[False, False]
)

lead_score_by_category.head(20)

,category,records,unique_users,avg_lead_score,high_intent_leads,avg_gmv,total_gmv,high_intent_rate
10,books_technical,267,259,94.36,255,87.56,23379.12,0.9551
67,tablets_printing_image,83,79,95.90,79,105.48,8754.61,0.9518
36,food,510,445,92.97,479,71.89,36664.44,0.9392
14,computers,203,179,91.48,190,1146.80,232799.43,0.9360
62,signaling_and_security,199,139,93.10,186,140.79,28017.05,0.9347
37,food_drink,278,225,94.54,259,70.82,19687.47,0.9317
24,drinks,379,292,92.31,353,74.33,28169.95,0.9314
8,books_general_interest,553,506,93.87,514,101.36,56052.40,0.9295
31,fashion_shoes,262,238,92.77,243,108.71,28481.64,0.9275
53,luggage_accessories,1092,1028,93.24,1011,156.48,170875.21,0.9258


In [22]:
lead_score_by_category.to_csv(TABLE_OUTPUT_DIR / "lead_score_by_category.csv", index=False)

print("lead_score_by_category.csv exported successfully.")

lead_score_by_category.csv exported successfully.


## Step 17: Lead Score by User Segment

This analysis evaluates whether high-value users have stronger lead scores and purchase intent.

It connects intent classification with user behaviour segmentation.

In [23]:
lead_score_by_user_segment = (
    intent_base
    .groupby("user_value_segment")
    .agg(
        records=("order_id", "count"),
        unique_users=("user_id", "nunique"),
        avg_lead_score=("lead_score", "mean"),
        high_intent_leads=("high_intent_flag", "sum"),
        avg_gmv=("gmv", "mean"),
        total_gmv=("gmv", "sum")
    )
    .reset_index()
)

lead_score_by_user_segment["avg_lead_score"] = lead_score_by_user_segment["avg_lead_score"].round(2)
lead_score_by_user_segment["avg_gmv"] = lead_score_by_user_segment["avg_gmv"].round(2)
lead_score_by_user_segment["total_gmv"] = lead_score_by_user_segment["total_gmv"].round(2)

lead_score_by_user_segment["high_intent_rate"] = (
    lead_score_by_user_segment["high_intent_leads"] / lead_score_by_user_segment["records"]
).round(4)

lead_score_by_user_segment

,user_value_segment,records,unique_users,avg_lead_score,high_intent_leads,avg_gmv,total_gmv,high_intent_rate
0,High Value,42766,31118,90.60,39592,244.47,10454977.90,0.9258
1,Low Value,32229,31120,91.97,29129,48.18,1552691.00,0.9038
2,Medium Value,35202,31120,91.52,32030,96.93,3412104.85,0.9099
3,Unknown,2453,2180,30.50,355,172.76,423779.49,0.1447


In [24]:
lead_score_by_user_segment.to_csv(TABLE_OUTPUT_DIR / "lead_score_by_user_segment.csv", index=False)

print("lead_score_by_user_segment.csv exported successfully.")

lead_score_by_user_segment.csv exported successfully.


## Step 18: Export Final LLM-Inspired Intent Dataset

The final intent classification dataset is exported as:

- `data/final/fact_reviews_llm.csv`

This table will be used in the next stages for:

1. Lead scoring model;
2. Recommendation strategy design;
3. A/B test simulation;
4. Tableau dashboard development.

In [26]:
# Select final output columns
fact_reviews_llm = intent_base[[
    "order_id",
    "user_id",
    "product_id",
    "seller_id",
    "category",
    "order_time",
    "price",
    "price_band",
    "gmv",
    "review_score",
    "review_text",
    "delivery_delay_days",
    "late_delivery_flag",
    "traffic_source",
    "device_type",
    "user_value_segment",
    "total_events",
    "viewed",
    "clicked",
    "added_to_cart",
    "inquired",
    "purchased",
    "sentiment",
    "intent_category",
    "purchase_intent",
    "lead_score",
    "high_intent_flag"
]].copy()

# Export final intent dataset
fact_reviews_llm.to_csv(FINAL_DIR / "fact_reviews_llm.csv", index=False)

print("fact_reviews_llm.csv exported successfully.")
print("File saved to:", FINAL_DIR / "fact_reviews_llm.csv")

fact_reviews_llm.csv exported successfully.
File saved to: ../data/final/fact_reviews_llm.csv


## Step 19: Summary of LLM-Inspired Intent Classification

In this notebook, I implemented a reproducible LLM-inspired intent classification workflow.

Key outputs generated:

- `fact_reviews_llm.csv`
- `intent_distribution.csv`
- `intent_category_summary.csv`
- `lead_score_by_category.csv`
- `lead_score_by_user_segment.csv`

Business value of this stage:

1. Converted unstructured review text into structured intent signals;
2. Classified users by sentiment and purchase intent;
3. Identified major intent categories such as delivery concern, price sensitivity, product quality concern, and ready-to-purchase signals;
4. Created an interpretable lead score for downstream recommendation strategy;
5. Created a high-intent lead flag for targeting and A/B test simulation.

This stage directly supports the recommendation strategy product workflow by surfacing user lead generation intent and preparing intent-aware features for downstream recommendation improvement.